# Evolving Cellular Automata with a Genetic Algorithm
### Neuromorphic Computing — Lab Tutorial

**Based on:** Mitchell, Crutchfield & Hraber (1994). *Evolving cellular automata to perform computations: Mechanisms and impediments.* Physica D, 75, 361–391.

---

**What you'll do in this tutorial (~15 minutes):**
1. Visualize how a 1D cellular automaton evolves over time
2. See the density classification task in action
3. Step through one generation of the genetic algorithm by hand
4. Run the full GA and watch fitness improve
5. Work through four experiments on your own

---

### Setup

**Option A — Google Colab (recommended, no install needed):**
Just click the Colab link shared by your instructor. Run the first code cell and you're ready.

**Option B — Run locally:**
```bash
pip install -r requirements.txt
python test_ca.py   # should print: All tests passed ✓
jupyter notebook tutorial.ipynb
```

In [ ]:
# ── Colab / local setup ────────────────────────────────────────────────────
# This cell detects whether you're in Google Colab and, if so, clones the
# course repo so ca.py, ga.py, etc. are available. If you're running locally
# just run: pip install -r requirements.txt  (then skip this cell).

import sys, os

if "google.colab" in sys.modules:
    import subprocess
    REPO = "https://github.com/melaniematunm/ca-ga-tutorial"  # ← update after creating your repo
    print("Cloning repo...")
    subprocess.run(["git", "clone", REPO, "ca_ga"], check=True, capture_output=True)
    os.chdir("ca_ga")
    print("Installing dependencies...")
    subprocess.run([sys.executable, "-m", "pip", "install", "-r",
                     "requirements.txt", "-q"], check=True)
    print("Colab setup complete ✓  (repo cloned, dependencies installed)")
else:
    print("Running locally ✓  (make sure you've run: pip install -r requirements.txt)")


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

# The two core modules in this project
from ca import apply_rule, run_ca, make_gkl_rule, get_lambda, evaluate_rule, generate_ics
from ga import (
    DEFAULT_CONFIG, random_population, evaluate_population,
    next_generation, compute_stats, single_point_crossover, mutate
)

rng = np.random.default_rng(42)
print("Imports OK ✓")

---
## Part 1 — What Is a 1D Cellular Automaton?

A **1D cellular automaton** is a row of cells, each either 0 (white) or 1 (black). At every time step, each cell looks at itself and its **r** nearest neighbours on each side, and updates its state according to a fixed **rule table**.

In this project:
- **r = 3** → each cell sees a 7-cell neighbourhood
- **2⁷ = 128** possible neighbourhood patterns → the rule table has **128 bits**
- The lattice wraps around (periodic boundary)
- We run for up to ~320 time steps and read the final state

The **spacetime diagram** below shows time flowing downward. Each row is one step.

In [ ]:
def plot_spacetime(rule, ic, max_steps=100, title="Spacetime diagram", ax=None):
    """Plot a CA spacetime diagram. Time flows downward."""
    history = [ic.copy()]
    config  = ic.copy()
    for _ in range(max_steps):
        if config.all() or not config.any():
            break
        config = apply_rule(config, rule)
        history.append(config.copy())
    grid = np.array(history)

    if ax is None:
        fig, ax = plt.subplots(figsize=(10, 5))
    ax.imshow(grid, cmap="binary", interpolation="nearest", aspect="auto")
    ax.set_xlabel("Cell index", fontsize=11)
    ax.set_ylabel("Time step", fontsize=11)
    ax.set_title(title, fontsize=12)
    return ax

# The GKL rule is a hand-crafted benchmark that already solves the task well
gkl = make_gkl_rule()

# Two example initial conditions — one low density, one high density
N = 149
ic_low  = np.zeros(N, dtype=np.uint8); ic_low[:40]  = 1; rng.shuffle(ic_low)
ic_high = np.zeros(N, dtype=np.uint8); ic_high[:90] = 1; rng.shuffle(ic_high)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
plot_spacetime(gkl, ic_low,  title=f"GKL rule — low density (ρ≈0.27 → all 0s)", ax=axes[0])
plot_spacetime(gkl, ic_high, title=f"GKL rule — high density (ρ≈0.60 → all 1s)", ax=axes[1])
plt.tight_layout()
plt.show()

print(f"GKL rule λ (fraction of 1-outputs): {get_lambda(gkl):.3f}")

---
## Part 2 — The Density Classification Task

**Goal:** Given a random initial configuration, the CA must determine whether the *majority* of cells started as 1 (density ρ > 0.5) or 0 (ρ < 0.5), then relax to the correct uniform state:

| Initial density | Correct final state |
|---|---|
| ρ < 0.5 | All cells → 0 |
| ρ > 0.5 | All cells → 1 |

This is surprisingly hard: the CA can only do *local* computation — each cell sees just 7 neighbours — yet it must solve a *global* problem (count the majority across 149 cells).

**Fitness** F_I is the fraction of I random initial conditions the rule classifies correctly. Random guessing ≈ 0.50. GKL ≈ 0.97 on a large test set.

In [ ]:
# Measure GKL accuracy on 200 random ICs
eval_rng  = np.random.default_rng(0)
ics_test  = generate_ics(200, N, eval_rng)
gkl_fit   = evaluate_rule(gkl, ics_test, max_steps=320)
print(f"GKL fitness on 200 ICs: {gkl_fit:.3f}")

# A random rule scores near 0.5 — it's just guessing
random_rule_eg = rng.integers(0, 2, 128, dtype=np.uint8)
rand_fit       = evaluate_rule(random_rule_eg, ics_test, max_steps=320)
print(f"Random rule fitness:    {rand_fit:.3f}")

---
## Part 3 — The Genetic Algorithm, Step by Step

The GA maintains a **population** of P=100 candidate rule tables (each a 128-bit array). Every generation:

```
1. EVALUATE  — score every rule on the same I random ICs (fitness = fraction correct)
2. SELECT    — keep the top E=20 rules unchanged (elitism)
3. REPRODUCE — fill remaining 80 slots by:
               (a) picking two elite parents at random
               (b) applying single-point crossover → two children
               (c) flipping m=2 random bits in each child (mutation)
4. REPEAT
```

The key insight from Mitchell et al.: because the **same ICs** are used for the whole population each generation, rules are compared fairly and selection pressure is consistent.

In [ ]:
# --- Step through ONE generation manually ---

cfg = {**DEFAULT_CONFIG, "I": 30, "M_mean": 150, "P": 100, "E": 20, "seed": 42}
step_rng = np.random.default_rng(42)

# Step 1: Create a random initial population
pop = random_population(cfg["P"], step_rng)
print(f"Population shape: {pop.shape}  (each row = one 128-bit rule)")
print(f"Example rule (first 16 bits): {pop[0, :16]}")
print(f"Example rule λ: {get_lambda(pop[0]):.3f}")

# Step 2: Evaluate fitness
fits = evaluate_population(pop, cfg, step_rng)
print(f"\nFitness range: {fits.min():.3f} – {fits.max():.3f}  (mean={fits.mean():.3f})")

# Step 3: Show what elite selection looks like
top_idx = np.argsort(-fits)[:cfg["E"]]
print(f"\nTop {cfg['E']} fitness scores: {fits[top_idx]}")

# Step 4: Crossover + mutation demo
parent_a, parent_b = pop[top_idx[0]], pop[top_idx[1]]
child_a, child_b   = single_point_crossover(parent_a, parent_b, step_rng)
child_a_mut        = mutate(child_a, m=2, rng=step_rng)

print(f"\nCrossover + mutation demo (first 20 bits):")
print(f"  Parent A: {parent_a[:20]}")
print(f"  Parent B: {parent_b[:20]}")
print(f"  Child A : {child_a[:20]}   (after crossover)")
print(f"  Child A*: {child_a_mut[:20]}  (after mutation — 2 bits flipped)")

---
## Part 4 — Run the Full GA

We'll run for **25 generations** with the fast default settings (I=30, M=150). This takes about 30–60 seconds on a laptop. Then we'll plot fitness and λ (the fraction of 1-outputs in the rule table) over time.

> **λ is a proxy for rule behaviour.** λ near 0.5 → the rule outputs 0 and 1 roughly equally, producing complex dynamics. λ near 0 or 1 → the rule is trivially all-0 or all-1.

In [ ]:
from ga import run_ga
import os, sys

# Run for 25 generations (fast settings)
results = run_ga(
    cfg={**DEFAULT_CONFIG, "G": 25, "I": 30, "M_mean": 150, "seed": 42},
    checkpoint_dir="checkpoints_tutorial",
    results_dir="results_tutorial",
    resume=False,
    use_crossover=True,
    verbose=True,
)

history  = results["history"]
best_rule = results["population"][np.argmax(results["fitnesses"])]

print(f"\nFinal best fitness: {results['fitnesses'].max():.3f}")
print(f"Best rule λ:        {get_lambda(best_rule):.3f}")

In [ ]:
# Plot fitness and λ history
gens        = [h["generation"] for h in history]
best_fit    = [h["best_fitness"]  for h in history]
mean_fit    = [h["mean_fitness"]  for h in history]
lambda_best = [h["best_lambda"]   for h in history]
lambda_mean = [h["mean_lambda"]   for h in history]

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 7), sharex=True)

ax1.plot(gens, best_fit,  lw=2,   label="Best rule")
ax1.plot(gens, mean_fit,  lw=1.5, ls="--", alpha=0.7, label="Population mean")
ax1.axhline(gkl_fit, color="red", ls=":", lw=1.5, label=f"GKL benchmark ({gkl_fit:.2f})")
ax1.set_ylabel("Fitness (F_30)", fontsize=11)
ax1.set_ylim(0.4, 1.05)
ax1.legend(fontsize=10)
ax1.set_title("GA Progress — Density Classification Task", fontsize=13)
ax1.grid(alpha=0.3)

ax2.plot(gens, lambda_best, lw=2,   label="Best rule λ")
ax2.plot(gens, lambda_mean, lw=1.5, ls="--", alpha=0.7, label="Population mean λ")
ax2.axhline(0.5, color="gray", ls=":", lw=1, label="λ = 0.5")
ax2.set_xlabel("Generation", fontsize=11)
ax2.set_ylabel("λ (fraction of 1-outputs)", fontsize=11)
ax2.set_ylim(0.3, 0.7)
ax2.legend(fontsize=10)
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.show()

# Show what the best evolved rule looks like
fig2, axes2 = plt.subplots(1, 2, figsize=(14, 5))
plot_spacetime(best_rule, ic_low,  title="Evolved rule — low density  (correct: → 0)",  ax=axes2[0])
plot_spacetime(best_rule, ic_high, title="Evolved rule — high density (correct: → 1)", ax=axes2[1])
plt.tight_layout()
plt.show()

### What to look for in the plots

- **Fitness jumps** — the GA doesn't improve smoothly; innovation happens in bursts. Mitchell et al. identified four distinct "epochs" of discovery across longer runs.
- **λ drift below 0.5** — successful rules tend to output more 0s than 1s. This symmetry-breaking appears early and is a signature of the GA finding useful structure.
- **Mean fitness lags the best** — the population takes a few generations to "catch up" after a good rule is discovered, as elitism spreads it through reproduction.

---
## Part 5 — Exercises

Each exercise is a small change to the code above. Run 25 generations each time and compare to the baseline run you just did.

> **Tip:** Keep your baseline results in memory — `results`, `history`, `best_fit`, etc. — so you can plot comparisons.

### Exercise 1 — Does the random seed matter?

Run the GA with a different seed (e.g. `seed=7`, `seed=123`). Does it reach the same final fitness? Does λ drift in the same direction? How many generations does it take to first cross fitness = 0.7?

*Hint: change only `seed` in the config dict below. Store your new run as `results2`.*

In [ ]:
# Exercise 1 — change the seed and re-run
results2 = run_ga(
    cfg={**DEFAULT_CONFIG, "G": 25, "I": 30, "M_mean": 150, "seed": 7},  # <-- try different seeds
    checkpoint_dir="checkpoints_ex1",
    results_dir="results_ex1",
    resume=False,
    use_crossover=True,
    verbose=False,   # suppress per-gen output for brevity
)

history2 = results2["history"]
best_fit2 = [h["best_fitness"] for h in history2]
gens2     = [h["generation"]   for h in history2]

plt.figure(figsize=(9, 4))
plt.plot(gens,  best_fit,  lw=2, label="Seed 42 (baseline)")
plt.plot(gens2, best_fit2, lw=2, label="Seed 7",    ls="--")
plt.xlabel("Generation"); plt.ylabel("Best fitness")
plt.title("Exercise 1: Does seed matter?")
plt.legend(); plt.grid(alpha=0.3); plt.tight_layout(); plt.show()

print(f"Seed 42 final best: {max(best_fit):.3f}")
print(f"Seed 7  final best: {max(best_fit2):.3f}")

### Exercise 2 — What does crossover actually contribute?

Run the GA with `use_crossover=False`. In this mode offspring are created by cloning an elite parent and applying mutation only — no gene mixing. Mitchell et al. (Section 11) found that crossover significantly increases the chance of reaching the higher "epochs" of performance.

*Questions to answer:*
- Is the final fitness lower, higher, or the same?
- Does the population converge faster or slower?
- What does this tell you about the role of crossover in exploring the search space?

In [ ]:
# Exercise 2 — turn off crossover
results_nocross = run_ga(
    cfg={**DEFAULT_CONFIG, "G": 25, "I": 30, "M_mean": 150, "seed": 42},
    checkpoint_dir="checkpoints_nocross",
    results_dir="results_nocross",
    resume=False,
    use_crossover=False,   # <-- the key change
    verbose=False,
)

history_nc  = results_nocross["history"]
best_fit_nc = [h["best_fitness"] for h in history_nc]
mean_fit_nc = [h["mean_fitness"] for h in history_nc]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))

ax1.plot(gens, best_fit,    lw=2, label="With crossover")
ax1.plot(gens, best_fit_nc, lw=2, label="No crossover", ls="--")
ax1.set_xlabel("Generation"); ax1.set_ylabel("Best fitness")
ax1.set_title("Best rule fitness"); ax1.legend(); ax1.grid(alpha=0.3)

ax2.plot(gens, mean_fit,    lw=2, label="With crossover")
ax2.plot(gens, mean_fit_nc, lw=2, label="No crossover", ls="--")
ax2.set_xlabel("Generation"); ax2.set_ylabel("Mean fitness")
ax2.set_title("Population mean fitness"); ax2.legend(); ax2.grid(alpha=0.3)

plt.suptitle("Exercise 2: Crossover vs. Mutation-only", fontsize=13)
plt.tight_layout(); plt.show()

### Exercise 3 — How important is population size?

Reduce the population from P=100 to P=20 (keeping E=4, so elite is still 20% of population). Does the GA still make progress? Does it stall or converge prematurely?

*This is a classic GA question: smaller populations are faster per generation but explore less of the search space.*

In [ ]:
# Exercise 3 — smaller population
results_small = run_ga(
    cfg={**DEFAULT_CONFIG, "G": 25, "I": 30, "M_mean": 150,
         "seed": 42, "P": 20, "E": 4},   # <-- P=20, elite=4 (still 20%)
    checkpoint_dir="checkpoints_small",
    results_dir="results_small",
    resume=False,
    use_crossover=True,
    verbose=False,
)

history_sm  = results_small["history"]
best_fit_sm = [h["best_fitness"] for h in history_sm]
mean_fit_sm = [h["mean_fitness"] for h in history_sm]

plt.figure(figsize=(9, 4))
plt.plot(gens, best_fit,    lw=2, label="P=100 (baseline)")
plt.plot(gens, best_fit_sm, lw=2, label="P=20", ls="--")
plt.xlabel("Generation"); plt.ylabel("Best fitness")
plt.title("Exercise 3: How much does population size matter?")
plt.legend(); plt.grid(alpha=0.3); plt.tight_layout(); plt.show()

### Exercise 4 — The GKL Benchmark: Can evolution match hand-design?

The GKL rule was *hand-crafted* by computer scientists who understood the task. The evolved rules are found by blind search. Evaluate both on a large test set and compare their spacetime behaviour on a *tricky* initial condition: one where ρ is very close to 0.5.

*Questions:*
- Which rule performs better on 500 ICs?
- On the near-50% IC, which rule is more confident? What does the spacetime diagram look like?
- What does this tell you about the relationship between structured design and evolutionary search?

In [ ]:
# Exercise 4 — GKL benchmark vs. evolved rule on a large test set
test_rng      = np.random.default_rng(999)
ics_large     = generate_ics(500, N, test_rng)

gkl_large_fit  = evaluate_rule(gkl,       ics_large, max_steps=320)
evol_large_fit = evaluate_rule(best_rule,  ics_large, max_steps=320)

print(f"GKL rule fitness on 500 ICs:     {gkl_large_fit:.3f}")
print(f"Evolved rule fitness on 500 ICs: {evol_large_fit:.3f}")

# Tricky IC: density just above 0.5 (75 ones out of 149)
ic_tricky = np.zeros(N, dtype=np.uint8)
ic_tricky[:75] = 1
rng.shuffle(ic_tricky)
print(f"\nTricky IC density: {ic_tricky.mean():.3f}  (correct answer: all 1s)")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
plot_spacetime(gkl,       ic_tricky, title="GKL rule — tricky IC (ρ≈0.50)",     ax=axes[0])
plot_spacetime(best_rule, ic_tricky, title="Evolved rule — tricky IC (ρ≈0.50)", ax=axes[1])
plt.suptitle("Exercise 4: Who handles near-50% density better?", fontsize=13)
plt.tight_layout(); plt.show()

---
## Stretch Exercise — Run Longer and Revisit

From the terminal:
```bash
python main.py --generations 100
```
or to resume from your 25-generation checkpoint:
```bash
python main.py --resume --generations 100 --checkpoint-dir checkpoints_tutorial
```

Then revisit Exercise 4 with the 100-generation evolved rule:
```python
import numpy as np
best_long = np.load("results/best_rule.npy")
```

**Discussion questions:**
1. After 100 generations, does the evolved rule approach GKL's performance?
2. Why might evolution find a *different* strategy than the GKL rule even at high fitness?
3. Mitchell et al. observed only ~92% of runs reach "Epoch 3" (fitness > 0.85 reliably). What does that imply about the fitness landscape?
4. How does this relate to neuromorphic computing? What parallel can you draw between the GA's search process and how biological neural circuits might emerge?

---
*Tutorial adapted from: Mitchell, Crutchfield & Hraber (1994), Physica D 75: 361–391.*  
*Code by Melanie M., UNM Complex AI Agents.*